In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_distances
import matplotlib.pyplot as plt

df= pd.read_csv("../data/BMW/sentiment_clean_reviews.csv")
print(df.shape)
print(df.head())

In [ ]:
df["sentiment"].value_counts()

In [ ]:
# Wir filtern zunächst die negativen Reviews heraus, um die Daten für die Issue Detection vorzubereiten
negative_reviews = df[df["sentiment"] == "negative"]
print("Number of negative reviews:", negative_reviews.shape[0])

In [ ]:
# Wir schauen uns die ersten Zeilen der negativen Reviews an, um einen Eindruck von den Daten zu bekommen
#negative_reviews[["content", "lemmatized_text"]].head()
negative_reviews[["content", "sentiment"]].head()

In [ ]:
#Für Embeddings nehmen wir den bereinigten Text, 
#da dieser bereits von Stoppwörtern und Sonderzeichen bereinigt wurde.
#Da wir jedoch einen Sentence Transformer verwenden, werden wir die cleaned_text Spalte verwenden, 
#da diese bereits in einem Format vorliegt, das für die Erstellung von Embeddings geeignet ist.
#Lemmatization wurde gemacht, um die Performance der Sentiment Modelle zu verbessern, 
#da sie die Anzahl der einzigartigen Wörter reduziert und somit die Generalisierung des Modells verbessert. 
#Für die Issue Detection hingegen ist es wichtig, 
#den Kontext und die spezifischen Formulierungen der Kunden zu erhalten, 
#um die Probleme besser zu identifizieren. 
# Daher verwenden wir hier den bereinigten Text, um die Embeddings zu erstellen.
texts= negative_reviews["content"].astype(str).tolist()
print(len(texts))

In [ ]:
#Wir prüfen die ersten 5 Einträge der Texte, um sicherzustellen, 
#dass sie korrekt bereinigt wurden und für die Erstellung von Embeddings geeignet sind.
texts[:5]

In [ ]:
#Sentence Transformer Embeddings erstellen
#Jeder Review wird in einen semantischen Vektor umgewandelt, der die Bedeutung des Textes einfängt.
from sentence_transformers import SentenceTransformer
#Modell laden (paraphrase-multilingual-MiniLM-L12-v2 ist ein kompaktes Modell, 
#das gut für die Erstellung von Embeddings geeignet ist). 
#Wir haben sowohl deutsche als auch englische Reviews, daher ist ein multilingual Modell sinnvoll.
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
#Embeddings erstellen
embeddings = model.encode(texts, show_progress_bar=True)
print(embeddings.shape)

In [ ]:
from sklearn.metrics import silhouette_score

inertia=[]
silhouette_scores= []
k_values= range(2, 15)
for k in k_values:
    kmeans= KMeans(n_clusters=k, random_state=42)
    cluster_labels= kmeans.fit_predict(embeddings)
    inertia.append(kmeans.inertia_)
    silhouette_avg= silhouette_score(embeddings, cluster_labels)
    silhouette_scores.append(silhouette_avg)

In [ ]:
#Hier machen wir den Elbow Plot, um die optimale Anzahl an Clustern zu bestimmen.
plt.figure(figsize=(8, 5))
plt.plot(k_values, inertia, marker="o")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method for Optimal k")
plt.show()

In [ ]:
#Nun machen wir den Silhouette Score Plot, um die optimale Anzahl an Clustern zu bestimmen.
plt.figure(figsize=(8, 5))
plt.plot(k_values, silhouette_scores, marker="o")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score for Optimal k")
plt.show()

##### Die optimale Anzahl an Clustern wurde mithilfe der Elbow-Methode und des Silhouette-Scores bestimmt. Der Elbow-Plot zeigt einen deutlichen Knick bei k=5, ab dem die Inertia nr noch geringfügig weiter abnimmt. Obwohl der Silhouette-Score bei k=2 seinen höchsten Wert erreicht, würde diese Clusteranzahl zu sehr breiten und wenig differenzierten Clustern führen.Daher wurde k=5 als ausgewogene Lösung gewählt, da diese Anzahl gut interpretierbare Issue-Kategorien ermöglicht und gleichzeitig eine angemessene Clustertrennung beibehält.

In [ ]:
#Clustering der Issues. 
# Hier verwenden wir K-Means, um die negativen Reviews in verschiedene Gruppen zu unterteilen,
#die jeweils ein bestimmtes Problem oder Thema repräsentieren.
from sklearn.cluster import KMeans
#Anzahl der Cluster festlegen (z.B. 5)
k=5
kmeans = KMeans(n_clusters=k, random_state=42)
#Clustering durchführen
cluster= kmeans.fit_predict(embeddings)
#Cluster-Zuordnungen der Reviews erhalten
negative_reviews["cluster"]= cluster
print(negative_reviews[["content", "cluster"]].head())

In [ ]:
#Threshold für die Erkennung neuer Probleme festlegen
distances_list =[]
for emb in embeddings:
    distances= cosine_distances([emb], kmeans.cluster_centers_)
    min_distance=distances.min()
    distances_list.append(min_distance)
distances_list= np.array(distances_list)

print("Average distance:", distances_list.mean())
print("Max distance:", distances_list.max())
DISTANCE_THRESHOLD= np.percentile(distances_list, 90)
print("Selected threshold:", DISTANCE_THRESHOLD)

In [ ]:
DISTANCE_THRESHOLD=0.55
print("Final threshold used:", DISTANCE_THRESHOLD)

In [ ]:
plt.hist(distances_list, bins=40)
plt.axvline(DISTANCE_THRESHOLD, color="red", linestyle="--")
plt.title("Distribution of Cosine Distances")
plt.xlabel("Distance to closest cluster")
plt.ylabel("Number of reviews")
plt.show()

##### UMAP-Visualisierung erstellen

In [ ]:
print(type(embeddings))
print(embeddings.shape)
print(np.isnan(embeddings).sum())
print(np.isinf(embeddings).sum())

In [ ]:
import umap
import numpy as np

embeddings=np.array(embeddings).astype(np.float32)
umap_model=umap.UMAP(n_neighbors=15, n_components=2, metric="cosine", random_state=42)
embedding_2d= umap_model.fit_transform(embeddings)
print(embedding_2d.shape)

In [ ]:
negative_reviews["x"]= embedding_2d[:, 0]
negative_reviews["y"]= embedding_2d[:, 1]

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 7))
scatter = plt.scatter(negative_reviews["x"], negative_reviews["y"], c=negative_reviews["cluster"], cmap="tab10", alpha=0.6)
plt.title("Issue Clusters in BMW App Reviews (UMAP)")
plt.xlabel("UMAP Dimension 1")
plt.ylabel("UMAP Dimension 2")
plt.colorbar(scatter, label="Cluster")
plt.show()

In [ ]:
#Wir schauen uns die Anzahl der Reviews in jedem Cluster an, um zu sehen, 
#wie die negativen Reviews gruppiert wurden.
negative_reviews.groupby("cluster").size()

In [ ]:
for i in range(k):
    print("\nCluster", i)
    sample= negative_reviews[negative_reviews["cluster"] == i]["content"].head(5)
    for review in sample:
        print("-", review)  

In [ ]:
#Code für wichtigste Wörter in jedem Cluster
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
#TF-IDF Vektorisierung der negativen Reviews
vectorizer = TfidfVectorizer(max_features=1000)
X = vectorizer.fit_transform(negative_reviews["content"])
terms= vectorizer.get_feature_names_out()
#Wichtigste Wörter pro Cluster identifizieren
for i in range(k):
    print("\nCluster", i)
    #Reviews im aktuellen Cluster auswählen
    cluster_indices = negative_reviews["cluster"] == i
    cluster_tfidf = X[cluster_indices.values]
    #Durchschnittliche TF-IDF Werte pro Wort im Cluster berechnen
    mean_tfidf = np.asarray(cluster_tfidf.mean(axis=0)).ravel()
    #Top 10 Wörter mit höchsten durchschnittlichen TF-IDF Werten auswählen
    top_words = mean_tfidf.argsort()[-10:][::-1]
    
    print([terms[j] for j in top_words])


In [ ]:
#Wir benennen jetzt die Cluster 
cluster_labels= {
    0: "Vehicle Connectivity Issues",
    1: "Login / Authentication Problems",
    2: "Feature / Software Support Issues",
    3: "App Functionality Problems",
    4: "Charging / Remote Services Issues"
}
negative_reviews["issues"]= negative_reviews["cluster"].map(cluster_labels)
print(negative_reviews[["content", "issues"]].head())

In [ ]:
#Wir zeigen nun welche Issues am häufigsten in den negativen Reviews genannt werden
issue_counts= negative_reviews["issues"].value_counts()
print(issue_counts)

In [ ]:
import matplotlib.pyplot as plt
issue_counts.plot(kind="bar")
plt.title("Most Common Issues in BMW App Reviews")
plt.xlabel("Issue Type")
plt.ylabel("Number of Reviews")
plt.xticks(rotation=90)
plt.show()

In [ ]:
#Clusterzentren speichern. Diese brauchen wir für New Issue Detection, 
#um neue Reviews den bestehenden Clustern zuzuordnen.
cluster_centers=kmeans.cluster_centers_
print(cluster_centers.shape)

In [ ]:
#Hier erstellen wir eine Funktion, die neue Reviews den bestehenden Clustern zuordnet,
#indem sie die Ähnlichkeit der neuen Review-Embeddings zu den Clusterzentren berechnet.
def detect_issue(review_text):
    #Embedding für die neue Review erstellen
    embedding = model.encode([review_text])
    #Ähnlichkeit zu den Clusterzentren berechnen
    distances = cosine_distances(embedding, kmeans.cluster_centers_)
    closest_cluster=distances.argmin()
    min_distance = distances.min()
    print("Min distance:", min_distance)
    if min_distance> DISTANCE_THRESHOLD: #Schwellenwert für die Zuordnung zu einem bestehenden Cluster (wurde rechnerisch ermittelt)
        return "New Issue Detected", None
    else:
        return "Known Issue", closest_cluster
    
    

In [ ]:
#Wenn ein neues Issue erkannt wird, erzeugen wir automatisch KEywords als Label.
def generate_issue_label(text):
    vector= vectorizer.transform([text])
    tfidf_scores=np.asarray(vector.todense()).flatten()
    top_indices= tfidf_scores.argsort()[-3:][::-1]
    words= [terms[i] for i in top_indices]
    return ", ".join(words)

In [ ]:
#Jetzt verbinden wir beide Funktionen 
def analyze_review(review_text):
    status, cluster_id = detect_issue(review_text)
    print("Detected cluster:", cluster_id)
    if status == "New Issue Detected":
        label = generate_issue_label(review_text)
        return {"status": "New Issue Detected", "Suggested_label": label}
    else:
        label= cluster_labels.get(int(cluster_id), "Other Issue")
        return {"status": "Known Issue", "issue_label": label}

In [ ]:
test_review= "The BMW app crashes every time I try to open vehicle status"
print(analyze_review(test_review))

In [ ]:
test_review= "Voice assistant in the BMW app is broken"
print(analyze_review(test_review))

In [ ]:
test_review= "Die BMW App stürzt jedes Mal ab, wenn ich versuche, den Fahrzeugstatus zu öffnen."
print(analyze_review(test_review))

In [ ]:
#Wir erstellen hiermit einen "models" Ordner
import os
os.makedirs("models", exist_ok=True)

In [ ]:
#Nun speichern wir die Modelle
import joblib
joblib.dump(kmeans, "models/kmeans_model.pkl")
#Threshold speichern
joblib.dump(DISTANCE_THRESHOLD, "models/distance_threshold.pkl")
#Cluster Labels speichern
joblib.dump(cluster_labels, "models/cluster_labels.pkl")
print("Models saved successfully")

In [ ]:
#-------------------------------------------------------------------------------------------

In [1]:
#Daten laden (clean & Standardisiert)
import pandas as pd

df = pd.read_csv("../data/BMW/clean_reviews.csv")

print(df.shape)
df.head()

(10150, 4)


,text,rating,date,sentiment
0,every time I tap on the widget to open the app...,3,2026-03-07 16:35:07,neutral
1,I have a vehicle with comfort access and a Sam...,2,2026-03-07 14:42:54,negative
2,Can't add my e34 and e39 :/,1,2026-03-07 09:50:37,negative
3,BMW stands for its reliability n performance i...,5,2026-03-07 07:47:33,positive
4,The lack of support for Octopus Intelligent Go...,2,2026-03-06 19:38:16,negative


In [2]:
#Nur relevante Reviews
df = df[df["rating"] <= 2].copy()
print("Negative reviews:", len(df))

Negative reviews: 4035


In [3]:
#Embeddings erzeugen (Kein TF-IDF hier. Wir nutzen Sentence Transformer)
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(df["text"].tolist(), show_progress_bar=True)

/Users/ayhan/Desktop/DS_Bootcamp/Capstone_Martin_Ayhan/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 127/127 [00:12<00:00, 10.16it/s]


In [4]:
#Clustering K-MEans
from sklearn.cluster import KMeans

k = 5  # später optimieren

kmeans = KMeans(n_clusters=k, random_state=42)
df["cluster"] = kmeans.fit_predict(embeddings)


In [5]:
#Wir speichern direkt
df["cluster"]


1        0
2        2
4        0
5        0
6        2
        ..
10137    1
10138    1
10141    3
10146    3
10148    1
Name: cluster, Length: 4035, dtype: int32

In [6]:
#Cluster prüfen
print(df["cluster"].value_counts())


cluster
1    1151
3     965
0     763
4     629
2     527
Name: count, dtype: int64


In [9]:
#Top Keywords pro Cluster
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import numpy as np

#Das wird ersetzt
#vectorizer = TfidfVectorizer(max_features=1000)
#X = vectorizer.fit_transform(df["text"])

#Durch das
german_stopwords = [
    "der","die","das","und","ist","ich","nicht","ein","eine","es","zu","den","im","für","mit"
]

all_stopwords = list(ENGLISH_STOP_WORDS.union(german_stopwords))

vectorizer = TfidfVectorizer(
    max_features=1000,
    stop_words=all_stopwords
)

X = vectorizer.fit_transform(df["text"])

terms = vectorizer.get_feature_names_out()

for i in range(k):
    print("\nCluster", i)
    
    cluster_indices = (df["cluster"] == i).values
    cluster_tfidf = X[cluster_indices]
    
    mean_tfidf = np.mean(cluster_tfidf, axis=0)
    top_words = np.argsort(mean_tfidf).tolist()[0][-10:]
    
    print([terms[j] for j in top_words])


Cluster 0
['does', 'use', 'new', 'update', 'work', 'doesn', 'vehicle', 'app', 'bmw', 'car']

Cluster 1
['auto', 'nur', 'auf', 'funktioniert', 'auch', 'kann', 'man', 'mehr', 'bmw', 'app']

Cluster 2
['car', 'error', 'time', 'update', 'log', 'keeps', 'doesn', 'login', 'working', 'work']

Cluster 3
['immer', 'keine', 'sich', 'man', 'kann', 'update', 'app', 'seit', 'funktioniert', 'mehr']

Cluster 4
['work', 'useless', 'working', 'login', 'log', 'use', 'update', 'time', 'car', 'app']


In [24]:
#Cluster automatisch benennen. Reviews pro Cluster sammeln
def get_cluster_samples(df, cluster_id):
    return df[df["cluster"] == cluster_id]["text"].tolist()

In [31]:
#LLM Prompt bauen
def generate_cluster_label(reviews):
    
    import ollama
    
    context = "\n".join(reviews)
    
    prompt = f"""
Return EXACTLY ONE issue label (2-3 words).
Choose the MOST COMMON issue across all reviews.

Rules:
- Only 2-3 words
- No punctuation
- Output must be a single line
- No explanation.
- No sentence.
- No list.

Examples:
Login Issue
App Crash
Connectivity Problem

Reviews:
{context}

Label:
"""
    
    response = ollama.chat(
        model="llama3",
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response["message"]["content"].strip()


In [32]:
#Für alle Cluster anwenden
cluster_labels = {}

for cluster_id in df["cluster"].unique():
    
    print(f"Processing cluster {cluster_id}...")
    
    samples = get_cluster_samples(df, cluster_id)
    
    label = generate_cluster_label(samples)
    
    cluster_labels[cluster_id] = label

Processing cluster 0...
Processing cluster 2...
Processing cluster 4...
Processing cluster 1...
Processing cluster 3...


In [33]:
#Labels ins DataFrame schreiben
df["cluster_label"] = df["cluster"].map(cluster_labels)

In [34]:
#Ergebnis checken
df[["text", "cluster", "cluster_label"]].head(10)

,text,cluster,cluster_label
1,"I have a vehicle with comfort access and a Samsung Galaxy S26 Ultra that has UWB, yet I just cannot set up my phone as a digital key. BMW support have confirmed that everything is supported by the car and in my country. So the problem is simply in the app that wouldn't properly support a car from 2020. How ridiculous is that for the premium we paid not only for the vehicle, but also for Connected Drive services.",0,"Based on the text, I would choose the label:\n\n**""DOESN'T WORK""**\n\nThis label captures the common theme of many users' experiences with the app, where they report issues such as the app not recognizing their car, failing to update vehicle status, or simply not functioning properly."
2,Can't add my e34 and e39 :/,2,"Based on your request, I've analyzed the issues mentioned by users and extracted one issue label that accurately describes the most common problem:\n\n**Login Issues**\n\nThis label encompasses various login-related problems, including:\n\n* Constantly being logged out\n* Unable to log in due to technical errors or invalid credentials\n* Issues with password recovery or reset\n* Problems connecting to the car's system\n\nThese issues were mentioned by many users throughout their comments."
4,The lack of support for Octopus Intelligent Go is frustrating. I can't understand why this has been removed; it's the most popular EV tariff in the UK.,0,"Based on the text, I would choose the label:\n\n**""DOESN'T WORK""**\n\nThis label captures the common theme of many users' experiences with the app, where they report issues such as the app not recognizing their car, failing to update vehicle status, or simply not functioning properly."
5,"To have my BMW charge overnight in a 'time slot', I have to preprogram a departure time. Why? Why should it be necessary to set a departure time before it will allow me to charge overnight? It makes no sense. Was the software designed by Franz Kafka?",0,"Based on the text, I would choose the label:\n\n**""DOESN'T WORK""**\n\nThis label captures the common theme of many users' experiences with the app, where they report issues such as the app not recognizing their car, failing to update vehicle status, or simply not functioning properly."
6,"Loss of push notifications issue is now fixed - thank you. Since the January update I am unable to schedule service items from nominated or other dealers - ""No services or repairs available. We are currently unable to retrieve available services or repair""",2,"Based on your request, I've analyzed the issues mentioned by users and extracted one issue label that accurately describes the most common problem:\n\n**Login Issues**\n\nThis label encompasses various login-related problems, including:\n\n* Constantly being logged out\n* Unable to log in due to technical errors or invalid credentials\n* Issues with password recovery or reset\n* Problems connecting to the car's system\n\nThese issues were mentioned by many users throughout their comments."
7,"Very bad experience, like these guys are from the Neaderthal compared to the current times. App goes în a loop when adding a new vehicle and bringing me to the same menu, without completing the add. Really really bad! Also a lot of other poorly developed functionalities. Long delay to activate car functions, sometimes no response. Bad.",0,"Based on the text, I would choose the label:\n\n**""DOESN'T WORK""**\n\nThis label captures the common theme of many users' experiences with the app, where they report issues such as the app not recognizing their car, failing to update vehicle status, or simply not functioning properly."
9,Changelog where?,2,"Based on your request, I've analyzed the issues mentioned by users and extracted one issue label that accurately describes the most common problem:\n\n**Login Issues**\n\nThis label encompasses various login-related problems, including:\n\n* Constantly being logged out\n* Unable to log in due to technical errors or inval

In [35]:
import pandas as pd
pd.set_option("display.max_colwidth", None)
df.head()

,text,rating,date,sentiment,cluster,cluster_label
1,"I have a vehicle with comfort access and a Samsung Galaxy S26 Ultra that has UWB, yet I just cannot set up my phone as a digital key. BMW support have confirmed that everything is supported by the car and in my country. So the problem is simply in the app that wouldn't properly support a car from 2020. How ridiculous is that for the premium we paid not only for the vehicle, but also for Connected Drive services.",2,2026-03-07 14:42:54,negative,0,"Based on the text, I would choose the label:\n\n**""DOESN'T WORK""**\n\nThis label captures the common theme of many users' experiences with the app, where they report issues such as the app not recognizing their car, failing to update vehicle status, or simply not functioning properly."
2,Can't add my e34 and e39 :/,1,2026-03-07 09:50:37,negative,2,"Based on your request, I've analyzed the issues mentioned by users and extracted one issue label that accurately describes the most common problem:\n\n**Login Issues**\n\nThis label encompasses various login-related problems, including:\n\n* Constantly being logged out\n* Unable to log in due to technical errors or invalid credentials\n* Issues with password recovery or reset\n* Problems connecting to the car's system\n\nThese issues were mentioned by many users throughout their comments."
4,The lack of support for Octopus Intelligent Go is frustrating. I can't understand why this has been removed; it's the most popular EV tariff in the UK.,2,2026-03-06 19:38:16,negative,0,"Based on the text, I would choose the label:\n\n**""DOESN'T WORK""**\n\nThis label captures the common theme of many users' experiences with the app, where they report issues such as the app not recognizing their car, failing to update vehicle status, or simply not functioning properly."
5,"To have my BMW charge overnight in a 'time slot', I have to preprogram a departure time. Why? Why should it be necessary to set a departure time before it will allow me to charge overnight? It makes no sense. Was the software designed by Franz Kafka?",2,2026-03-05 08:46:07,negative,0,"Based on the text, I would choose the label:\n\n**""DOESN'T WORK""**\n\nThis label captures the common theme of many users' experiences with the app, where they report issues such as the app not recognizing their car, failing to update vehicle status, or simply not functioning properly."
6,"Loss of push notifications issue is now fixed - thank you. Since the January update I am unable to schedule service items from nominated or other dealers - ""No services or repairs available. We are currently unable to retrieve available services or repair""",2,2026-03-04 01:13:00,negative,2,"Based on your request, I've analyzed the issues mentioned by users and extracted one issue label that accurately describes the most common problem:\n\n**Login Issues**\n\nThis label encompasses various login-related problems, including:\n\n* Constantly being logged out\n* Unable to log in due to technical errors or invalid credentials\n* Issues with password recovery or reset\n* Problems connecting to the car's system\n\nThese issues were mentioned by many users throughout their comments."


In [36]:
print(cluster_labels)

{0: 'Based on the text, I would choose the label:\n\n**"DOESN\'T WORK"**\n\nThis label captures the common theme of many users\' experiences with the app, where they report issues such as the app not recognizing their car, failing to update vehicle status, or simply not functioning properly.', 2: "Based on your request, I've analyzed the issues mentioned by users and extracted one issue label that accurately describes the most common problem:\n\n**Login Issues**\n\nThis label encompasses various login-related problems, including:\n\n* Constantly being logged out\n* Unable to log in due to technical errors or invalid credentials\n* Issues with password recovery or reset\n* Problems connecting to the car's system\n\nThese issues were mentioned by many users throughout their comments.", 4: 'Here are some of the most common issue labels that can be extracted from your text:\n\n1. **Crashes**: The app crashes frequently, especially when opening or trying to use certain features.\n2. **Login

In [ ]:
#OPTIONAL
cluster_names = {
    0: "Connectivity Issues",
    1: "Login Problems",
    2: "Charging Issues",
    3: "App Crashes",
    4: "Performance Issues"
}

df["cluster_label"] = df["cluster"].map(cluster_names)

In [37]:
#Speichern
df.to_csv("../data/BMW/reviews_with_clusters.csv", index=False)

In [ ]:
#----------------------------------------------------------------------------

In [ ]:
#Cluster automatisch benennen. Reviews pro Cluster sammeln
def get_cluster_samples(df, cluster_id, n=10):
    return df[df["cluster"] == cluster_id]["text"].sample(n=min(n, len(df[df["cluster"] == cluster_id]))).tolist()

In [ ]:
#LLM Prompt bauen
def generate_cluster_label(reviews):
    
    import ollama
    
    context = "\n".join(reviews)
    
    prompt = f"""
You are a UX expert.

Analyze the following user reviews and identify the main issue.

Reviews:
{context}

Return ONLY a short issue label (2-4 words).

Example:
"Login Issues"
"Connectivity Problems"
"Charging Errors"
"""
    
    response = ollama.chat(
        model="llama3",
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response["message"]["content"].strip()


In [ ]:
#Für alle Cluster anwenden
cluster_labels = {}

for cluster_id in df["cluster"].unique():
    
    print(f"Processing cluster {cluster_id}...")
    
    samples = get_cluster_samples(df, cluster_id)
    
    label = generate_cluster_label(samples)
    
    cluster_labels[cluster_id] = label

In [ ]:
#Labels ins DataFrame schreiben
df["cluster_label"] = df["cluster"].map(cluster_labels)


In [ ]:
#Ergebnis checken
df[["text", "cluster", "cluster_label"]].head(10)


In [ ]:
#optional labels specihern-----------------------------------------------------------

In [ ]:
import pickle

with open("../models/cluster_labels.pkl", "wb") as f:
    pickle.dump(cluster_labels, f)